In [ ]:
builder.add_node("reflect", reflect_on_evaluation)
builder.add_node("re_evaluate", re_evaluate_answer)

builder.add_edge("evaluate", "reflect")
builder.add_conditional_edges(
    "reflect",
    lambda state: state.get("next_step", ""),
    {
        "re_evaluate": "re_evaluate",
        "decide_next_step": "decide"
    },
)

builder.add_edge("re_evaluate", "reflect")



In [ ]:
def evaluate_answer(state: InterviewState) -> InterviewState:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    prompt_evaluate = ChatPromptTemplate.from_template("""
    당신은 면접관입니다. 아래의 질문과 지원자의 답변을 보고 평가하세요.

    평가 기준:
    1. 질문과의 연관성:
       - 상(우수): 질문의 핵심 의도에 정확히 부합하며, 전반적인 내용을 명확히 다룸
       - 중(보통): 질문과 관련은 있으나 핵심 포인트가 부족하거나 부분적으로 누락됨
       - 하(미흡): 질문과 관련이 약하거나 엉뚱한 내용을 중심으로 함

    2. 답변의 구체성:
       - 상(우수): 구체적인 사례, 수치, 프로젝트 내용 등으로 뒷받침됨
       - 중(보통): 사례나 근거는 있으나 구체성이 다소 부족함
       - 하(미흡): 모호하고 추상적인 답변

    --- 질문 ---
    {current_question}

    --- 지원자 답변 ---
    {current_answer}

    JSON 형식으로만 평가 결과를 작성하세요. 코드블록(```)은 사용하지 마세요.
    {{
      "질문과의 연관성": "상/중/하 중 하나",
      "답변의 구체성": "상/중/하 중 하나",
      "총평": "전반적인 평가 요약 (1~2문장)"
    }}


    """)

    #  실행
    chain = prompt_evaluate | llm
    response = chain.invoke({
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", "")
    })

    result_text = response.content.strip()

    # JSON 파싱
    try:
        evaluation = json.loads(result_text)
    except Exception:
        evaluation = {
            "질문과의 연관성": "평가 실패",
            "답변의 구체성": "평가 실패",
            "총평": result_text
        }

    # 대화 로그 및 평가 추가
    conversation = state.get("conversation", [])
    conversation.append({
        "role": "human",
        "content": state.get("current_answer", "")
    })
    conversation.append({
        "role": "ai",
        "content": f"평가 결과: {evaluation}"
    })

    evaluation_list = state.get("evaluation", [])
    evaluation_list.append({
        "question": state.get("current_question", ""),
        "answer": state.get("current_answer", ""),
        "question_strategy": state.get("current_strategy", ""), # question_strategy 추가
        "evaluation": evaluation
    })

    return {
        **state,
        "conversation": conversation,
        "evaluation": evaluation_list,
    }


In [ ]:
def reflect_on_evaluation(state: InterviewState) -> InterviewState:
    """LLM을 이용해 최근 평가가 적절한지 되돌아보는 Reflection 노드"""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    last_eval = state.get("evaluation", [])[-1] if state.get("evaluation") else {}
    eval_fields = last_eval.get("evaluation", {})
    question = state.get("current_question", "")
    answer = state.get("current_answer", "")

    prompt = ChatPromptTemplate.from_template("""
    당신은 AI 면접 평가 품질 관리관입니다.

    아래의 질문, 답변, 기존 평가를 검토하여 평가가 적절한지 판단하세요.
    만약 평가가 부정확하거나 보완이 필요하다면 그 이유를 구체적으로 말하고,
    최종 판단을 "정상" 또는 "재평가 필요" 중 하나로 내리세요.

    --- 질문 ---
    {question}

    --- 지원자 답변 ---
    {answer}

    --- 기존 평가 ---
    {evaluation}

    JSON 형식으로만 응답하세요:
    {{
      "판단": "정상" 또는 "재평가 필요",
      "이유": "판단 근거를 1~2문장으로 서술"
    }}
    """)

    chain = prompt | llm
    response = chain.invoke({
        "question": question,
        "answer": answer,
        "evaluation": json.dumps(eval_fields, ensure_ascii=False)
    })

    result_text = response.content.strip()

    try:
        reflection = json.loads(result_text)
    except Exception:
        reflection = {"판단": "재평가 필요", "이유": "LLM 응답 파싱 실패"}

    state["reflection_status"] = reflection.get("판단", "재평가 필요")
    state["reflection_feedback"] = reflection.get("이유", "")

    state["next_step"] = (
        "re_evaluate" if state["reflection_status"] == "재평가 필요"
        else "decide_next_step"
    )

    return state


In [ ]:
def re_evaluate_answer(state: InterviewState) -> InterviewState:

    """
    Reflection 결과가 '재평가 필요'일 때 실행되는 재평가 노드.
    기존 evaluate_answer()와 유사하지만, 재평가 이력을 남기고
    이전 평가를 참고하여 보완 평가를 수행한다.
    """
    re_eval_count = state.get("re_eval_count", 0) + 1
    state["re_eval_count"] = re_eval_count

#재평가 2회까지만
    if re_eval_count > 2:
        state["reflection_status"] = "정상"
        state["reflection_feedback"] = "재평가 한도를 초과하여 평가를 확정합니다."
        state["next_step"] = "decide_next_step"
        return state

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    last_eval = state.get("evaluation", [])[-1] if state.get("evaluation") else {}
    prev_evaluation = last_eval.get("evaluation", {})

    prompt = ChatPromptTemplate.from_template("""
    당신은 AI 면접관입니다. 아래는 이전 평가 결과와 새로운 답변입니다.
    이전 평가가 미흡했다고 판단되어 재평가를 수행합니다.

    이전 평가:
    {prev_evaluation}

    --- 질문 ---
    {current_question}

    --- 지원자 답변 ---
    {current_answer}

    JSON 형식으로만 재평가 결과를 작성하세요. 코드블록(```)은 사용하지 마세요.
    {{
      "질문과의 연관성": "상/중/하 중 하나",
      "답변의 구체성": "상/중/하 중 하나",
      "총평": "보완된 평가 요약 (1~2문장)"
    }}
    """)

    chain = prompt | llm
    response = chain.invoke({
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", ""),
        "prev_evaluation": json.dumps(prev_evaluation, ensure_ascii=False)
    })

    result_text = response.content.strip()

    # JSON 파싱
    try:
        new_evaluation = json.loads(result_text)
    except Exception:
        new_evaluation = {
            "질문과의 연관성": "평가 실패",
            "답변의 구체성": "평가 실패",
            "총평": result_text
        }

    # 평가 리스트에 새 평가 추가
    evaluation_list = state.get("evaluation", [])
    evaluation_list.append({
        "question": state.get("current_question", ""),
        "answer": state.get("current_answer", ""),
        "evaluation": new_evaluation,
        "re_evaluation": True  # 재평가 여부 표시
    })

    # 다음 단계는 다시 Reflection으로
    state["next_step"] = "reflect"

    return {
        **state,
        "evaluation": evaluation_list,
    }